# 01 資料準備

讀檔 → 資料更新檢查 → 結構驗證 → CODiS 特殊值處理 → 去除重複 → 五都篩選 → 對齊 10 分鐘頻率 → 預報版本。

| | 內容 |
|---|---|
| 輸入 | `data/raw/`（負載、CODiS、Accuweather、Windy、電價日曆表）、`config/` |
| 輸出 | `data/processed/load_clean.parquet`、`targets.parquet`、`weather_observed_daily.parquet`、`weather_forecast_daily.parquet`；`output/reports/01_*.csv` |

所有處理都呼叫 `src/`，與比賽當天的 `run_submission.py` 是同一套函式。
資料截止日由 `config/settings.py` 的 `DATA_AVAILABLE_END` 決定，讀檔時由程式過濾。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config").is_dir():   # 找到專案根目錄（notebook 可從任何位置執行）
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import os
import polars as pl
from src.logging_setup import setup_logging

os.chdir(ROOT)
setup_logging(stage="notebook_01", level="WARNING")
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

In [2]:
from config import paths, settings
from src.data import accuweather, checks, external, imputer, loader, manifest, weather
from src import workflow

paths.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print("資料截止日：", settings.DATA_AVAILABLE_END, "　作業時點：", settings.OPERATION_TIME)

資料截止日： 2026-06-30 　作業時點： 08:30


## 1. 資料更新檢查

以目前檔案重建 manifest，與版控中的 `data/manifest.json` 逐檔比對（雜湊、筆數、時間範圍、欄位）；
有唯一鍵的檔再與上次的內容快照**逐筆**比對。歷史區段有變動只警告、不中止。
（這裡只檢查，不接受；接受由 `run_submission.py` 在成功產檔後進行。）

In [3]:
check = manifest.update_check()
status = pl.DataFrame([{"檔案": f["path"], "狀態": f["status"]} for f in check["files"]])
print("歷史區段有變動：", check["history_changed"])
status.group_by("狀態").len()

歷史區段有變動： False


狀態,len
str,u32
"""unchanged""",14


In [4]:
status.filter(pl.col("狀態") != "unchanged")

檔案,狀態
str,str


## 2. 讀檔與結構驗證

每個讀檔函式都會檢查必要欄位、型別、時間格點與連續性，不符即中止並列出差異
（負載不容許任何缺口；外生氣象容許 `EXOGENOUS_MAX_GAP_HOURS` 小時）。

In [5]:
load = loader.load_raw_load()
calendar_df = external.load_calendar()
forecast_hourly = accuweather.load_station_hourly()
pl.DataFrame([
    {"資料": "負載", "列數": load.height, "起": str(load["ts"].min()), "迄": str(load["ts"].max()),
     "缺值": load["Load_MW"].null_count()},
    {"資料": "電價日曆表", "列數": calendar_df.height, "起": str(calendar_df["date"].min()),
     "迄": str(calendar_df["date"].max()), "缺值": 0},
    {"資料": "Accuweather（五站逐小時）", "列數": forecast_hourly.height,
     "起": str(forecast_hourly["Date"].min()), "迄": str(forecast_hourly["Date"].max()),
     "缺值": forecast_hourly["AirTemperature"].null_count()},
])

資料,列數,起,迄,缺值
str,i64,str,str,i64
"""負載""",131328,"""2024-01-01 00:00:00""","""2026-06-30 23:50:00""",6
"""電價日曆表""",18263,"""2011-01-01""","""2060-12-31""",0
"""Accuweather（五站逐小時）""",118180,"""2024-01-01 00:00:00""","""2026-09-18 23:00:00""",0


In [6]:
accuweather.coverage_report(forecast_hourly)

stn_ID,天數,列數,氣溫缺值,UV缺值
str,u32,u32,u32,u32
"""466881""",986,23636,0,11608
"""466920""",986,23636,0,11641
"""467410""",986,23636,0,11583
"""467441""",986,23636,0,11579
"""467490""",986,23636,0,11589


## 3. CODiS 特殊值

CODiS 以**數值**編碼特殊狀態（`…f` 旗標欄全期為空）。處理規則集中在 `config/settings.py` 的 `CODIS_*`：

| 原始值 | 處理 |
|---|---|
| 缺測碼（−9.5、−9.95、−99.x、−999.x、−9995、−9999.x） | 缺值 |
| 降水 −9.8（雨跡） | 0.1 mm，另設 `Precipitation_Accumulation_trace` = 1 |
| 未登記且 ≤ −90 的碼、無法轉數值的字串 | 缺值並警告 |

每一種轉換的筆數如下（同時寫入 `output/reports/01_codis_special_values.csv`）。

In [7]:
raw_observed = pl.read_csv(paths.WEATHER_FILE, infer_schema_length=0)
cleaned, report = weather.clean_observations(raw_observed)
report.write_csv(paths.REPORTS_DIR / "01_codis_special_values.csv")
report.sort("筆數", descending=True)

欄位,原始值,處理,筆數
str,str,str,i64
"""SoilTemperatureAt5cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8885
"""SoilTemperatureAt20cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8743
"""SoilTemperatureAt30cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8725
"""SoilTemperatureAt100cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8716
"""SoilTemperatureAt10cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8714
"""SoilTemperatureAt0cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8695
"""SoilTemperatureAt50cm_Instantaneous""","""-99.5""","""缺測碼 → 缺值""",8695
"""Precipitation_Accumulation""","""-9.8""","""雨跡 → 0.1""",7903
"""UVIndex_Accumulation""","""-99.95""","""缺測碼 → 缺值""",720


In [8]:
report.group_by("處理").agg(pl.col("筆數").sum()).sort("筆數", descending=True)

處理,筆數
str,i64
"""缺測碼 → 缺值""",62004
"""雨跡 → 0.1""",7903


## 4. 去除重複

完全重複的列刪除並記錄；鍵相同但數值不同**不自行挑選**，會中止並列出範例。
Accuweather 與 Windy 的鍵是「地點（或機組）＋目標時刻」——兩者都**沒有發布時間欄**，
同一目標時刻只有一個版本。

In [9]:
sources = {
    "負載": (pl.read_csv(paths.LOAD_DATA_FILE, infer_schema_length=0), ["Date_Time"]),
    "CODiS": (raw_observed, ["Date", "stn_ID"]),
    "Accuweather（五站）": (forecast_hourly, ["Date", "stn_ID"]),
    "Windy": (pl.read_csv(paths.WINDY_FILE, infer_schema_length=0), ["unit_name", "forecast_time"]),
}
rows = []
for name, (frame, key) in sources.items():
    kept = checks.deduplicate(frame, key, name)
    rows.append({"資料": name, "唯一鍵": " + ".join(key), "原始列數": frame.height,
                 "刪除完全重複": frame.height - kept.height, "鍵同值異": 0})
duplicates = pl.DataFrame(rows)
duplicates.write_csv(paths.REPORTS_DIR / "01_duplicates.csv")
duplicates

2026-09-29 13:28:51 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


資料,唯一鍵,原始列數,刪除完全重複,鍵同值異
str,str,i64,i64,i64
"""負載""","""Date_Time""",131328,0,0
"""CODiS""","""Date + stn_ID""",109440,0,0
"""Accuweather（五站）""","""Date + stn_ID""",118180,0,0
"""Windy""","""unit_name + forecast_time""",45877,122,0


## 5. 五都與測站對應

氣象只用五都：CODiS 沒有桃園的代表測站，故不含桃園。
Accuweather 對照表的「北區」「中正區」在多個縣市重名，**必須用縣市 + 鄉鎮兩欄配對**。

In [10]:
names = {code: name for name, code in settings.WEATHER_STATIONS.items()}
pl.DataFrame([
    {"城市": names[code], "CODiS 測站": code, "Accuweather location_key": key}
    for code, key in settings.ACCUWEATHER_STATIONS.items()
])

城市,CODiS 測站,Accuweather location_key
str,str,str
"""新北""","""466881""","""2515462"""
"""臺北""","""466920""","""2515405"""
"""臺中""","""467490""","""2515419"""
"""臺南""","""467410""","""2515406"""
"""高雄""","""467441""","""2515417"""


## 6. 對齊 10 分鐘頻率與補值，建立每日 6 目標

補齊 10 分鐘格點（缺時間戳補空值，不刪列）→ 補值 → 每日 6 目標 → 併入電價日曆的日別。
比賽當天換上正式資料後，`run_submission.py` 第 4 步做的就是這一段。

In [11]:
imputed, daily = workflow.build_processed()
print("10 分鐘序列：", imputed.shape, "　每日表：", daily.shape)
imputer.summarize_imputation(imputed)

2026-09-29 13:28:51 | WARNING  | src.workflow | 有缺漏的日子：
shape: (3, 3)
┌────────────┬──────────┬───────────┐
│ date       ┆ n_points ┆ n_missing │
│ ---        ┆ ---      ┆ ---       │
│ date       ┆ u32      ┆ u32       │
╞════════════╪══════════╪═══════════╡
│ 2025-03-19 ┆ 144      ┆ 4         │
│ 2025-07-15 ┆ 144      ┆ 1         │
│ 2026-05-23 ┆ 144      ┆ 1         │
└────────────┴──────────┴───────────┘


2026-09-29 13:28:52 | WARNING  | src.data.external | 合併後 solar_term 有 852 / 912 列缺值（93.4%）


10 分鐘序列： (131328, 4) 　每日表： (912, 27)


impute_method,n_points,n_days
str,u32,u32
"""daytype_profile""",4,1
"""linear""",2,2


## 7. 預報版本

預報只能用作業時點前已發布的版本。Accuweather 與 Windy 的歷史檔**沒有發布時間欄**，
每個（地點, 目標時刻）只有一筆——無法挑版本，也無法量測前置期誤差。

因此回測一律用 **honest 模式**：歷史用 CODiS 觀測、目標日用 Accuweather 預報
（以預測起點以前的重疊期逐站校正）。歷史檔裡的預報很可能接近觀測，比賽當天提前
兩天多發布的預報誤差會更大——honest 模式的分數是**下限估計**。

In [12]:
versions = forecast_hourly.group_by("Date", "stn_ID").len()["len"].max()
print("Accuweather 每個（測站, 目標時刻）的版本數上限：", versions)

Accuweather 每個（測站, 目標時刻）的版本數上限： 1


## 8. 輸出給下一本

In [13]:
from src.features import accuweather as accuweather_features

observed_daily = external.load_weather()
forecast_daily = accuweather_features.build_forecast()
observed_daily.write_parquet(paths.PROCESSED_DIR / "weather_observed_daily.parquet")
forecast_daily.write_parquet(paths.PROCESSED_DIR / "weather_forecast_daily.parquet")
print("觀測：", observed_daily.shape, "　預報：", forecast_daily.shape)

觀測： (912, 21) 　預報： (986, 21)
